<div dir="rtl" style="text-align:right; font-family:'Segoe UI', 'Arial Hebrew', Arial, sans-serif; background: linear-gradient(135deg, #1864ab 0%, #4dabf7 100%); color:white; padding:24px 28px; border-radius:12px; margin-bottom:18px;">

<h1 style="color:white; margin:0 0 6px;">DBSCAN &mdash; שאלות למרצה</h1>

<div style="font-size:15px; opacity:0.95;">הקשר: פרויקט זיהוי אנומליות בתעבורת רשת מבוסס Wireshark. מעבר מ-DBSCAN בלבד ל-HDBSCAN עם Hopkins statistic ו-fallback ל-DBSCAN.</div>

</div>


<div dir="rtl" style="background:#fff8e1; border-right:5px solid #f59f00; border-radius:10px; padding:18px 22px; margin:8px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.75; font-size:15px; color:#222;">

<div style="font-size:18px; font-weight:700; color:#b8540a; margin-bottom:8px;">שאלה #1 &mdash; האם DBSCAN בכלל המודל הנכון?</div>

ה-DBSCAN שלי בוחר <code>eps</code> אוטומטית מ-<b>k-distance elbow</b> ויוצא <b>0.78</b> בסשן אחד ו-<b>4.86</b> בסשן אחר על אותה רשת. שניהם מקבצים את כל ה-IP-ים ל-<b>cluster יחיד</b> + מעט noise. ה-<code>Silhouette</code> לא מוגדר במצב כזה.<br><br>בנתונים האלה &mdash; האם DBSCAN בכלל המודל הנכון, או שהפיצ'רים שלי לא מפרידים מספיק טוב והייתי צריך לעבור ל-<b>HDBSCAN / GMM / Mean-Shift</b>?

</div>


<div dir="rtl" style="font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; font-size:13px; color:#666; margin-bottom:4px; text-align:right;">הקוד הרלוונטי &mdash; הקלאסטרינג + בדיקת Silhouette:</div>


In [ ]:
dbscan = DBSCAN(eps=eps_auto, min_samples=2)
ip_agg["cluster"] = dbscan.fit_predict(X)

_labels   = ip_agg["cluster"].values
_nonnoise = _labels != -1
_n_clusters = int(len(set(_labels[_nonnoise])))

if _nonnoise.sum() >= 2 and _n_clusters >= 2:
    _sil = float(silhouette_score(X[_nonnoise], _labels[_nonnoise]))
else:
    _sil = None    # Silhouette undefined for a single cluster


<div dir="rtl" style="background:#e8f4fd; border-right:4px solid #1c7ed6; border-radius:8px; padding:12px 18px; margin:6px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:14px; color:#1a1a1a;">

<b style="color:#1864ab;">תשובה צפויה:</b> אם המטרה היא לזהות חריגים (anomalies) &mdash; cluster יחיד + noise זה ממצא תקף ברשת רגילה. אם רוצים אשכולות התנהגותיים מובחנים &mdash; <b>HDBSCAN</b> עדיף (לא דורש eps, מטפל בצפיפויות משתנות). GMM ייכשל כי תעבורת רשת skewed. מומלץ קודם לחשב <b>Hopkins statistic</b> כדי לדעת אם בכלל יש מבנה לאשכל.

</div>


<div dir="rtl" style="background:#fff8e1; border-right:5px solid #f59f00; border-radius:10px; padding:18px 22px; margin:8px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.75; font-size:15px; color:#222;">

<div style="font-size:18px; font-weight:700; color:#b8540a; margin-bottom:8px;">שאלה #2 &mdash; <code>eps</code> דינמי לכל סשן &mdash; האם זה לגיטימי?</div>

הקוד מחשב את <code>eps</code> לכל סשן בנפרד: <code>k-distance</code> עם <code>k=2</code>, מציאת ה-<b>מרפק</b> (נגזרת שנייה מינימלית), עיגול ל-2 ספרות.<br><br>התוצאה: <b>0.78 ב-S1, 4.86 ב-S2</b> &mdash; שני ערכי סף שונים לחלוטין על אותה רשת.<br><br>האם זה לגיטימי לכייל <code>eps</code> נפרד לכל סשן? מצד אחד זה מתאים את ההגדרה לצפיפות הנתונים &mdash; מצד שני זה אומר שאני <b>לא יכול להשוות חד-משמעית בין S1 ל-S2</b> כי הם נחתכים בסף שונה.

</div>


<div dir="rtl" style="font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; font-size:13px; color:#666; margin-bottom:4px; text-align:right;">הקוד הרלוונטי &mdash; אוטומציה של <code>eps</code> ע"י k-distance elbow:</div>


In [ ]:
k = 2
nbrs = NearestNeighbors(n_neighbors=k).fit(X)
distances, _ = nbrs.kneighbors(X)
k_dist = np.sort(distances[:, k-1])[::-1]

if len(k_dist) >= 4:
    d1 = np.diff(k_dist)
    d2 = np.diff(d1)
    elbow_idx = int(np.argmin(d2)) + 1
    eps_auto  = float(round(k_dist[elbow_idx], 2))
else:
    eps_auto = 1.3    # fallback when too few points


<div dir="rtl" style="background:#e8f4fd; border-right:4px solid #1c7ed6; border-radius:8px; padding:12px 18px; margin:6px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:14px; color:#1a1a1a;">

<b style="color:#1864ab;">תשובה צפויה:</b> לגיטימי <b>עם תיעוד מפורש</b>. שלוש אסכולות בקהילה: (א) eps נפרד adaptive לכל סשן, (ב) eps אחיד לכל הסשנים להשוואה ישירה, (ג) היברידי &mdash; להריץ עם שניהם ולדווח. הפרקטיקה המקובלת: להשתמש ב-eps נפרד אבל להשוות רק <b>מאפיינים אגרגטיביים</b> (כמה noise, כמה clusters), לא תוויות מספריות.

</div>


<div dir="rtl" style="background:#fff8e1; border-right:5px solid #f59f00; border-radius:10px; padding:18px 22px; margin:8px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.75; font-size:15px; color:#222;">

<div style="font-size:18px; font-weight:700; color:#b8540a; margin-bottom:8px;">שאלה #3 &mdash; ערך ה-<code>fallback</code> של 1.3 &mdash; מאיפה הוא?</div>

כשיש פחות מ-4 IP-ים, הקוד עושה <code>fallback</code> ל-<code>eps=1.3</code>. האם הערך הזה אקראי או שיש לו ביסוס? יש דרך אקדמית טובה יותר לבחור fallback?

</div>


<div dir="rtl" style="font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; font-size:13px; color:#666; margin-bottom:4px; text-align:right;">הקוד הרלוונטי &mdash; ה-fallback של 1.3:</div>


In [ ]:
if len(k_dist) >= 4:
    d1 = np.diff(k_dist)
    d2 = np.diff(d1)
    elbow_idx = int(np.argmin(d2)) + 1
    eps_auto  = float(round(k_dist[elbow_idx], 2))
else:
    eps_auto = 1.3    # <-- fallback value


<div dir="rtl" style="background:#e8f4fd; border-right:4px solid #1c7ed6; border-radius:8px; padding:12px 18px; margin:6px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:14px; color:#1a1a1a;">

<b style="color:#1864ab;">תשובה צפויה:</b> ערך אמפירי שמתאים למרחב שעבר <code>StandardScaler</code> &mdash; אחרי נירמול רוב המרחקים בין נקודות נורמליות נופלים בטווח <code>[0.5, 2.5]</code>, ו-1.3 הוא חציון סביר. רלוונטי רק לקצוות (פחות מ-4 IP-ים) שבהם ממילא אין מה לאשכל. <b>Ester et al. (1996)</b> ממליצים על <b>percentile 90 של k-distance</b> כברירת מחדל אקדמית מבוססת יותר.

</div>


<div dir="rtl" style="background:#fff8e1; border-right:5px solid #f59f00; border-radius:10px; padding:18px 22px; margin:8px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.75; font-size:15px; color:#222;">

<div style="font-size:18px; font-weight:700; color:#b8540a; margin-bottom:8px;">שאלה #4 &mdash; <code>Silhouette = n/a</code> &mdash; מה המדד החלופי?</div>

ב-Model Diagnostics רואים <code>Silhouette = n/a</code> כי הוא לא מוגדר מתמטית ל-<b>cluster יחיד</b> (אין "אשכול אחר" לחשב אליו את <code>b(i)</code>).<br><br>אני עדיין רוצה למדוד <b>איכות ההפרדה</b> בין cluster ל-noise. מה המדד הנכון? <b>DBCV</b>? <b>Davies-Bouldin</b>? משהו אחר?<br><br>וגם &mdash; האם cluster יחיד עם 6 noise points הוא ממצא משמעותי או "כישלון של המודל"?

</div>


<div dir="rtl" style="font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; font-size:13px; color:#666; margin-bottom:4px; text-align:right;">הקוד הרלוונטי &mdash; חישוב Silhouette מותנה:</div>


In [ ]:
try:
    if _nonnoise.sum() >= 2 and _n_clusters >= 2:
        _sil = float(silhouette_score(X[_nonnoise], _labels[_nonnoise]))
    else:
        _sil = None    # mathematically undefined for a single cluster
except Exception:
    _sil = None


<div dir="rtl" style="background:#e8f4fd; border-right:4px solid #1c7ed6; border-radius:8px; padding:12px 18px; margin:6px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:14px; color:#1a1a1a;">

<b style="color:#1864ab;">תשובה צפויה:</b> <b>DBCV (Density-Based Clustering Validation)</b> &mdash; המדד הנכון לקלאסטרינג מבוסס-צפיפות. בניגוד ל-Silhouette הוא <b>לוקח noise בחשבון</b> ומודד יחס בין צפיפות פנימית ובין-אשכולית. Davies-Bouldin / Calinski-Harabasz דורשים 2+ אשכולות &mdash; לא יעזרו כאן.<br><br>cluster יחיד + 6 noise points הוא <b>ממצא משמעותי</b>, לא כישלון &mdash; ברשת רגילה רוב התעבורה דומה, וה-noise הם החריגים האמיתיים. הוכחה נדרשת: <b>Hopkins &gt; 0.7</b> וקרוס-וולידציה של ה-noise עם חוקים דטרמיניסטיים.

</div>


<div dir="rtl" style="background:#fff8e1; border-right:5px solid #f59f00; border-radius:10px; padding:18px 22px; margin:8px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.75; font-size:15px; color:#222;">

<div style="font-size:18px; font-weight:700; color:#b8540a; margin-bottom:8px;">שאלה #5 &mdash; האם המעבר ל-HDBSCAN + Hopkins נכון תיאורטית?</div>

הוספתי לקוד שני שינויים: <b>Hopkins statistic</b> (בדיקה האם בכלל יש מבנה אשכולי), ו-<b>HDBSCAN</b> כקלאסטרינג ראשי עם fallback ל-DBSCAN.<br><br>הרציונל שלי: HDBSCAN לא דורש <code>eps</code> &mdash; הוא מטפל ב<b>צפיפויות משתנות</b> שזה מאפיין מובהק של תעבורת רשת (IoT בדפוס אחד, browsers בדפוס שני). Hopkins מספק "סנדק" לכל ניסיון אשכול.<br><br>האם הגישה הזאת מקובלת אקדמית? <b>איזה ערך סף של Hopkins</b> נחשב הוכחה מובהקת לקיום מבנה אשכולי?

</div>


<div dir="rtl" style="font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; font-size:13px; color:#666; margin-bottom:4px; text-align:right;">הקוד הרלוונטי &mdash; Hopkins statistic + בחירת HDBSCAN/DBSCAN:</div>


In [ ]:
def hopkins_statistic(X, m=None, random_state=42):
    """H ~ 0.5 = random; H > 0.7 = cluster tendency."""
    rng = np.random.default_rng(random_state)
    n, d = X.shape
    if m is None: m = max(5, int(0.1 * n))
    idx     = rng.choice(n, size=m, replace=False)
    sample  = X[idx]
    mins, maxs = X.min(axis=0), X.max(axis=0)
    synth   = rng.uniform(mins, maxs, size=(m, d))
    nbrs = NearestNeighbors(n_neighbors=2).fit(X)
    w = nbrs.kneighbors(sample, n_neighbors=2)[0][:, 1]
    u = nbrs.kneighbors(synth,  n_neighbors=1)[0][:, 0]
    return float(u.sum() / (u.sum() + w.sum()))

H = hopkins_statistic(X)

# Primary: HDBSCAN if available
if HDBSCAN_AVAILABLE and X.shape[0] >= 5:
    hdb = hdbscan.HDBSCAN(min_cluster_size=max(3, int(0.03 * X.shape[0])),
                          min_samples=2)
    labels = hdb.fit_predict(X)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 1:
        ip_agg["cluster"] = labels    # HDBSCAN succeeded
    else:
        ip_agg["cluster"] = run_dbscan_fallback(X)
else:
    ip_agg["cluster"] = run_dbscan_fallback(X)


<div dir="rtl" style="background:#e8f4fd; border-right:4px solid #1c7ed6; border-radius:8px; padding:12px 18px; margin:6px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:14px; color:#1a1a1a;">

<b style="color:#1864ab;">תשובה צפויה:</b> <b>גישה נכונה</b>. הספרות (Campello et al., 2013) מציגה HDBSCAN בדיוק לתרחיש של צפיפויות משתנות. הסף המקובל ל-Hopkins: <b>H &gt; 0.75</b> נחשב הוכחה חזקה; <b>0.5-0.75</b> אזור אפור; <b>≈ 0.5</b> אין מבנה בכלל. השילוב של Hopkins כ-"שומר סף" + HDBSCAN כקלאסטרינג + DBSCAN כ-fallback &mdash; זוהי גישה אקדמית <b>מקובלת ומבוססת</b>. שווה לציטוט: <i>Lawson &amp; Jurs (1990)</i> ל-Hopkins, <i>Campello, Moulavi &amp; Sander (2013)</i> ל-HDBSCAN.

</div>


<div dir="rtl" style="background:#fff8e1; border-right:5px solid #f59f00; border-radius:10px; padding:18px 22px; margin:8px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.75; font-size:15px; color:#222;">

<div style="font-size:18px; font-weight:700; color:#b8540a; margin-bottom:8px;">שאלה #6 &mdash; הערכה אקדמית בלי תוויות אמת</div>

הפרויקט שלי לא-מפוקח &mdash; אין לי <code>ground truth</code> ולכן <b>אני לא יכול למדוד precision / recall</b>. ההערכה שלי מסתמכת על: <b>Hopkins statistic</b>, <b>Silhouette</b> (כשמוגדר), ויחס clusters-to-noise.<br><br>אם הייתי רוצה לחזק את ההערכה &mdash; האם <b>הזרקת התקפות סינתטיות ידועות</b> (<code>port scan</code>, <code>DNS tunneling</code>) לתוך ה-PCAP, כדי ליצור <code>ground truth</code> נקודתי &mdash; זה <b>מקובל אקדמית</b>?

</div>


<div dir="rtl" style="font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; font-size:13px; color:#666; margin-bottom:4px; text-align:right;">הקוד הרלוונטי &mdash; הערכת איכות פנימית בלי תוויות:</div>


In [ ]:
H   = hopkins_statistic(X)      # cluster tendency [0,1]
_sil = (silhouette_score(X[_nonnoise], _labels[_nonnoise])
        if _n_clusters >= 2 else None)

S["_hopkins"]    = H
S["_silhouette"] = _sil
S["_n_clusters"] = _n_clusters
S["_n_noise"]    = _n_noise
# All evaluation metrics here are internal — no ground-truth labels.


<div dir="rtl" style="background:#e8f4fd; border-right:4px solid #1c7ed6; border-radius:8px; padding:12px 18px; margin:6px 0 14px; font-family: 'Segoe UI', 'Arial Hebrew', Arial, sans-serif; line-height:1.7; font-size:14px; color:#1a1a1a;">

<b style="color:#1864ab;">תשובה צפויה:</b> כן, בהחלט. הפרקטיקה נקראת <b>semi-synthetic ground truth injection</b> והיא הסטנדרט בקהילה. ה-datasets המובילים נוצרו בדיוק כך: <b>CIC-IDS2017</b>, <b>UNSW-NB15</b> &mdash; תעבורה רגילה + הזרקה מבוקרת. כלים סטנדרטיים: <b>Scapy</b> להזרקת packets, <b>nmap</b> ל-port scans, <b>dnscat2</b> ל-DNS tunneling.<br><br>מומלץ לציטוט: <b>Sharafaldin et al. (2018)</b>, <b>Ring et al. (2019)</b>.

</div>
